# Pre-process Amazon Reviews 2023

Same pipeline as `prepare_amazon.py` (dedup → rating filter → k-core → map ids → leave-one-out split → product table → simulator jsonl), adapted for the 2023 schema: `.jsonl.gz` instead of `.json.gz`, join on `parent_asin`, flat `categories`, list-valued `description`.

# 0. Import & logging

In [1]:
import os
import gzip
import json
import pickle
import logging
from local_package.config.data import AMAZON_RAW_DIR, AMAZON_PROCESSED_DIR
from local_package.config.log import setup_logging, AMAZON_PROCESS_LOG_DIR
import pandas as pd

In [2]:
logger = setup_logging(name="process", level=logging.INFO, to_file=True, log_dir=AMAZON_PROCESS_LOG_DIR)

# 1. Configuration

In [3]:
# Seed
SEED = 2024

In [4]:
# Path
CATEGORY = "All_Beauty"  # matches the filenames below; swap for any category from the dataset table

DATA_DIR = AMAZON_RAW_DIR / CATEGORY

REVIEW_FILE = DATA_DIR / f"{CATEGORY}.jsonl.gz"
META_FILE = DATA_DIR / f"meta_{CATEGORY}.jsonl.gz"

OUTPUT_DIR = AMAZON_PROCESSED_DIR / CATEGORY / "chatbot"  # where train/valid/test/products/simulator files go
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
# Rating threshold for filtering reviews
RATING_THRESHOLD = 3.0  # rows below this rating are dropped

# Min interactions per user/item for k-core filtering
USER_K = 5  # k-core: min interactions per user
ITEM_K = 5  # k-core: min interactions per item

# Maximum lengths for simulator strings
MAX_HISTORY_LEN = 10  # max past items shown in a simulator history string
MAX_TITLE_LEN = 50  # max chars of a product title used in simulator strings
MAX_DESCRIPTION_SENTENCES = 2  # how many description list entries to join into one string

# Simulator sampling
SIMULATOR_SAMPLE_N = 900  # test users sampled for the simulator jsonl export

In [6]:
# Columns used from the raw data
USED_COL = {
    "review": ["user_id", "parent_asin", "rating", "timestamp"],
    "meta": ["parent_asin", "title", "category", "price", "description"],
}

# 2. Load raw data

In [7]:
def iter_jsonl_gz(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

In [8]:
def load_reviews_and_meta(data_dir, review_file, meta_file):
    reviews_tsv = os.path.join(data_dir, "reviews.tsv")
    meta_tsv = os.path.join(data_dir, "meta.tsv")

    if not (os.path.exists(reviews_tsv) and os.path.exists(meta_tsv)):
        logger.info("No tsv cache found, parsing raw jsonl.gz files")
        review_df = pd.DataFrame.from_records(iter_jsonl_gz(review_file))
        meta_df = pd.DataFrame.from_records(iter_jsonl_gz(meta_file))
        review_df.to_csv(reviews_tsv, index=None, sep="|")
        meta_df.to_csv(meta_tsv, index=None, sep="|")
    else:
        logger.info("Loading from cached tsv files")
        review_df = pd.read_csv(reviews_tsv, sep="|", low_memory=False)
        meta_df = pd.read_csv(meta_tsv, sep="|", low_memory=False)

    logger.info("Shape of reviews: %s", review_df.shape)
    logger.info("Shape of meta: %s", meta_df.shape)
    return review_df, meta_df

In [9]:
review_df, meta_df = load_reviews_and_meta(DATA_DIR, REVIEW_FILE, META_FILE)

2026-09-06 13:57:40 [INFO] process: Loading from cached tsv files
2026-09-06 13:57:54 [INFO] process: Shape of reviews: (701528, 10)
2026-09-06 13:57:54 [INFO] process: Shape of meta: (112590, 14)


# 3. Clean metadata

In [10]:
def join_description(x, max_sentences=MAX_DESCRIPTION_SENTENCES):
    # description comes back from the tsv cache as a stringified list; eval it, then join
    if isinstance(x, str) and x.startswith("["):
        x = eval(x)
    if isinstance(x, list):
        return " ".join(x[:max_sentences]) if x else "No description"
    return "No description"

In [11]:
meta_df = meta_df[~meta_df["title"].isna()].reset_index(drop=True)
logger.info("Meta after dropping missing titles: %s", meta_df.shape)

2026-09-06 13:57:54 [INFO] process: Meta after dropping missing titles: (112578, 14)


In [12]:
meta_df["description"] = meta_df["description"].apply(join_description)
meta_df["categories"] = meta_df["categories"].apply(lambda x: eval(x) if isinstance(x, str) and x.startswith("[") else [])
meta_df["category"] = meta_df["categories"].apply(lambda x: x[0] if x else "Unknown")

In [13]:
# Column selection and renaming for consistency
review_df = review_df[USED_COL["review"]].rename(columns={"parent_asin": "item_id"})
meta_df = meta_df[USED_COL["meta"]].rename(columns={"parent_asin": "item_id"})

# 4. Filter out reviews and chat interactions/simulations

In [14]:
def get_valid_ids(df, col_name, k):
    frequency = df.groupby([col_name])[[col_name]].count()
    return frequency[frequency[col_name] >= k].index

In [15]:
def keep_first_filter(df, user_col="user_id", item_col="item_id", time_col="timestamp"):
    logger.info("Keeping first interaction per duplicated review, begin: %s", df.shape)
    df = df.sort_values(by=[user_col, time_col]).reset_index(drop=True)
    df = df.drop_duplicates(subset=[user_col, item_col], keep="first").reset_index(drop=True)
    logger.info("After keep-first filter: %s", df.shape)
    return df

In [16]:
def low_rating_filter(df, rating_thres=RATING_THRESHOLD, rating_col="rating"):
    logger.info("Filtering ratings below %.1f, begin: %s", rating_thres, df.shape)
    df = df[df[rating_col] >= rating_thres].reset_index(drop=True)
    logger.info("After rating filter: %s", df.shape)
    return df

In [17]:
def k_core_filter(df, user_k=USER_K, item_k=ITEM_K, user_col="user_id", item_col="item_id", max_iter=20):
    logger.info("k-core filtering (user_k=%d, item_k=%d), begin: %s", user_k, item_k, df.shape)
    num_users_prev, num_items_prev = len(df[user_col].unique()), len(df[item_col].unique())
    delta, it = True, 0
    while delta and it < max_iter:
        valid_users = get_valid_ids(df, user_col, user_k)
        df = df[df[user_col].isin(valid_users)]
        valid_items = get_valid_ids(df, item_col, item_k)
        df = df[df[item_col].isin(valid_items)]
        num_users, num_items = len(valid_users), len(valid_items)
        delta = (num_users != num_users_prev) or (num_items != num_items_prev)
        logger.info("Iter %d: users %d/%d, items %d/%d", it, num_users, num_users_prev, num_items, num_items_prev)
        num_users_prev, num_items_prev = num_users, num_items
        it += 1
    logger.info("After k-core filter: %s", df.shape)
    return df

In [18]:
review_df = review_df[review_df["item_id"].isin(meta_df["item_id"])].reset_index(drop=True)
data_df = keep_first_filter(review_df)
data_df = low_rating_filter(data_df)
data_df = k_core_filter(data_df).reset_index(drop=True)

2026-09-06 13:57:58 [INFO] process: Keeping first interaction per duplicated review, begin: (701444, 4)
2026-09-06 13:58:00 [INFO] process: After keep-first filter: (693847, 4)
2026-09-06 13:58:00 [INFO] process: Filtering ratings below 3.0, begin: (693847, 4)
2026-09-06 13:58:00 [INFO] process: After rating filter: (550401, 4)
2026-09-06 13:58:00 [INFO] process: k-core filtering (user_k=5, item_k=5), begin: (550401, 4)
2026-09-06 13:58:01 [INFO] process: Iter 0: users 1196/503638, items 644/97176
2026-09-06 13:58:01 [INFO] process: Iter 1: users 365/1196, items 425/644
2026-09-06 13:58:01 [INFO] process: Iter 2: users 277/365, items 357/425
2026-09-06 13:58:01 [INFO] process: Iter 3: users 238/277, items 318/357
2026-09-06 13:58:01 [INFO] process: Iter 4: users 220/238, items 298/318
2026-09-06 13:58:01 [INFO] process: Iter 5: users 207/220, items 284/298
2026-09-06 13:58:01 [INFO] process: Iter 6: users 200/207, items 280/284
2026-09-06 13:58:01 [INFO] process: Iter 7: users 198/200,

# 5. Mapping users and items' IDs

In [19]:
def map_id(df, user_colname="user_id", item_colname="item_id"):
    logger.info("Mapping user and item ids to contiguous integers")
    users, items = df[user_colname].unique(), df[item_colname].unique()
    user_map = {u: k + 1 for k, u in enumerate(users)}
    item_map = {i: k + 1 for k, i in enumerate(items)}
    df[user_colname] = df[user_colname].apply(lambda x: user_map[x])
    df[item_colname] = df[item_colname].apply(lambda x: item_map[x])
    return df, user_map, item_map

In [20]:
data_df, user_map, item_map = map_id(data_df)

2026-09-06 13:58:01 [INFO] process: Mapping user and item ids to contiguous integers


In [21]:
# Save the id maps to a json file for later use
with open(os.path.join(OUTPUT_DIR, "map.json"), "w") as f:
    json.dump({"item": item_map, "user": user_map}, f)
logger.info("Saved id maps to %s", os.path.join(OUTPUT_DIR, "map.json"))

2026-09-06 13:58:01 [INFO] process: Saved id maps to C:\Users\i_am_fuch\Desktop\agentic-rag-for-rcm-sys\data\processed\amazon\All_Beauty\chatbot\map.json


# 6. Leave-one-out split

In [22]:
def split_leave_one_out_seq(data, col_name, time_colname, col_names_2_return):
    df_sorted = data.sort_values(by=[col_name, time_colname]).reset_index(drop=True)
    df_test = df_sorted.groupby(by=col_name, as_index=False).nth(-1)
    df_train = df_sorted.iloc[df_sorted.index.difference(df_test.index)]
    return (
        df_train.reset_index(drop=True)[col_names_2_return],
        df_test.reset_index(drop=True)[col_names_2_return],
    )

In [23]:
df_train_0, df_test = split_leave_one_out_seq(data_df, "user_id", "timestamp", ["user_id", "item_id", "timestamp"])
df_train, df_valid = split_leave_one_out_seq(df_train_0, "user_id", "timestamp", ["user_id", "item_id"])

In [24]:
df_train.to_csv(os.path.join(OUTPUT_DIR, "train.tsv"), index=None)
df_valid.to_csv(os.path.join(OUTPUT_DIR, "valid.tsv"), index=None)
df_test.to_csv(os.path.join(OUTPUT_DIR, "test.tsv"), index=None)
df_train_0.to_csv(os.path.join(OUTPUT_DIR, "user_history.tsv"), index=None)

logger.info(
    "Saved splits to %s (train=%d, valid=%d, test=%d, full_history=%d)",
    OUTPUT_DIR, len(df_train), len(df_valid), len(df_test), len(df_train_0),
)

2026-09-06 13:58:01 [INFO] process: Saved splits to C:\Users\i_am_fuch\Desktop\agentic-rag-for-rcm-sys\data\processed\amazon\All_Beauty\chatbot (train=1492, valid=198, test=198, full_history=1690)


# 7. Product table

In [25]:
saved_meta_df = meta_df[meta_df["item_id"].isin(item_map.keys())]
saved_meta_df = saved_meta_df.drop_duplicates(subset=["item_id"], keep="first").reset_index(drop=True)
saved_meta_df["item_id"] = saved_meta_df["item_id"].apply(lambda x: item_map[x])

In [26]:
user_history = df_train_0.groupby("user_id").agg(list)
item_count = user_history["item_id"].explode().value_counts()
saved_meta_df.rename(columns={"item_id": "id"}, inplace=True)
saved_meta_df["visited_num"] = saved_meta_df["id"].apply(lambda x: item_count.loc[x] if x in item_count else 0)

In [27]:
saved_meta_df.to_feather(os.path.join(OUTPUT_DIR, "products.ftr"))
saved_meta_df.to_csv(os.path.join(OUTPUT_DIR, "products.csv"), index=None, sep="|")
logger.info("Saved product table (%d items) to %s", len(saved_meta_df), OUTPUT_DIR)

2026-09-06 13:58:01 [INFO] process: Saved product table (280 items) to C:\Users\i_am_fuch\Desktop\agentic-rag-for-rcm-sys\data\processed\amazon\All_Beauty\chatbot


# 8. Simulator jsonl sample

In [28]:
def write_jsonl(obj, fpath):
    try:
        with open(fpath, "w") as outfile:
            for entry in obj:
                json.dump(entry, outfile)
                outfile.write("\n")
        logger.info("Saved %d records to %s", len(obj), fpath)
    except Exception as e:
        fallback = f"{fpath}.pkl"
        logger.exception("Failed to write jsonl (%s), falling back to pickle at %s", e, fallback)
        with open(fallback, "wb") as tempfile:
            pickle.dump(obj, tempfile)

In [29]:
saved_meta_df_indexed = saved_meta_df.set_index("id")
id2title = {id_: saved_meta_df_indexed.loc[id_].title[:MAX_TITLE_LEN] for id_ in saved_meta_df_indexed.index}

In [30]:
n_sample = min(SIMULATOR_SAMPLE_N, len(df_test))
test_data = df_test.sample(n_sample, random_state=SEED)
test_data["history"] = test_data["user_id"].apply(
    lambda x: "; ".join([id2title[i] for i in user_history.loc[x]["item_id"][-MAX_HISTORY_LEN:]])
)
test_data["target"] = test_data["item_id"].apply(lambda x: saved_meta_df_indexed.loc[x].title)
test_data.reset_index(drop=True, inplace=True)

In [31]:
simulator_path = os.path.join(OUTPUT_DIR, f"simulator_test_data_{n_sample}.jsonl")
write_jsonl(test_data[["history", "target"]].to_dict("records"), simulator_path)
logger.info("Pipeline complete.")

2026-09-06 13:58:02 [INFO] process: Saved 198 records to C:\Users\i_am_fuch\Desktop\agentic-rag-for-rcm-sys\data\processed\amazon\All_Beauty\chatbot\simulator_test_data_198.jsonl
2026-09-06 13:58:02 [INFO] process: Pipeline complete.
